# Diagnosis LI-Small: `batch_accumulation 1` + paralel 2 GPU

Pada uji protokol tesis, baseline LI belum konvergen dalam 20 epoch (F1 test Multi-GIN 22,93%, MEGA-GIN 33,00%; paper 32,21 / 43,67; Multi-GIN baru mulai belajar di epoch 12). Notebook ini menguji dua hal sekaligus:

1. **`optim.batch_accumulation 1`** (config paper = 4): batch efektif 2048, langkah optimizer 4x lebih banyak per epoch dengan biaya komputasi yang sama. Apakah model LI konvergen lebih cepat dan F1 mendekati paper?
2. **Paralel 2 GPU**: Multi-GIN di GPU 0 dan MEGA-GIN di GPU 1 dalam satu sesi. Apakah waktu per epoch tetap mendekati run berurutan (363 / 352 detik)? Jika ya, kuota GPU untuk 110 run hampir separuh.

Selain itu protokol sama dengan `configs/thesis/` (maks. 20 epoch, early stopping 5, test sekali). Di akhir, metadata analisis dibuat ulang dengan log lengkap (di run LI sebelumnya file metadata tidak terbentuk).

**Cara menjalankan:**
1. Settings: Input dataset IBM AML, **Accelerator: GPU T4 x2** (wajib dua GPU), **Internet On**.
2. **Save Version -> Save & Run All (Commit)**. Perkiraan: ~2,5-3 jam jika paralel berjalan baik; maksimal ~5,5 jam jika ternyata tidak ada percepatan.
3. Unduh `results_diag.zip` dari Output, simpan ke `project_aml/hasil_kaggle/diag_LI/`.

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/aanpujianto070990/MEGA-GNN-TMLR.git"
REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

assert os.path.isdir(f"{REPO_DIR}/MegaGNN"), "Clone gagal -- pastikan Settings -> Internet = On, lalu jalankan ulang sel ini."
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
import sys, torch

print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f"gpu {i}:", torch.cuda.get_device_name(i))
if N_GPU < 2:
    print("PERINGATAN: hanya", N_GPU, "GPU -- pilih Accelerator 'GPU T4 x2'. Run akan dijalankan berurutan.")

TORCH_MM = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
CUDA_TAG = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
PYG_WHL = f"https://data.pyg.org/whl/torch-{TORCH_MM}+{CUDA_TAG}.html"

!pip install -q torch_geometric yacs ogb gdown tensorboardX torchmetrics pybind11
!pip install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {PYG_WHL}
# genagg sengaja non-editable (lihat Pilot_Reproduksi_Kaggle.ipynb).
!pip uninstall -y -q genagg
!pip install -q --no-deps {REPO_DIR}/genagg

!cd {REPO_DIR}/MegaGNN/cpp && python setup.py build_ext --inplace > build.log 2>&1; ls *.so
!cd {REPO_DIR} && python -c "import MegaGNN, torch_scatter, torch_sparse; from MegaGNN.cpp import ports_cpp; print('Import MegaGNN OK')"

## 2. Data dan cache praproses

Cache LI dibangun **sekali** sebelum dua run dimulai, supaya kedua proses tidak memproses dan menulis file yang sama bersamaan.

In [ ]:
import glob

DATA_DIR = "/tmp/aml_data"
OUT_DIR = "/kaggle/working/results_diag"
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = {"Small-HI": "HI-Small_Trans.csv", "Small-LI": "LI-Small_Trans.csv"}
PATTERNS = {"Small-HI": "HI-Small_Patterns.txt", "Small-LI": "LI-Small_Patterns.txt"}
RAW = {}
for name, fname in DATASETS.items():
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if not matches:
        print(f"PERINGATAN: {fname} tidak ditemukan di /kaggle/input")
        continue
    RAW[name] = matches[0]
    raw_dir = f"{DATA_DIR}/AML/{name}/raw"
    os.makedirs(raw_dir, exist_ok=True)
    if not os.path.exists(f"{raw_dir}/{fname}"):
        os.symlink(matches[0], f"{raw_dir}/{fname}")
    print(f"{name}: {matches[0]}")

!cd {REPO_DIR} && python thesis/prepare_data.py --dir {DATA_DIR} --name Small-LI

## 3. Jalankan dua run paralel

Output tiap run ditulis ke `stdout_<model>_gpu<i>.txt`; sel ini mencetak status epoch terakhir setiap 10 menit. Run yang sudah punya `run_summary.json` dilewati.

In [ ]:
import json, subprocess, time
import numpy as np
import pandas as pd

DATASET, SEED, TAG = "Small-LI", 1, "diag-acc1"
OVERRIDES = ["optim.batch_accumulation", "1"]
JOBS = [("Multi-GIN", 0), ("MEGA-GIN", 1 if N_GPU >= 2 else 0)]


def run_dir_of(model, gpu):
    # Tiap proses hanya melihat satu GPU (CUDA_VISIBLE_DEVICES) dan memakainya sebagai cuda:0,
    # jadi MEGA selalu menamai foldernya -gpu0; nama model membedakan kedua run.
    return f"{OUT_DIR}/AML-{DATASET}-{model}-{TAG}-gpu0/{SEED}"


def launch(model, gpu):
    cmd = ["python", "-u", "-m", "MegaGNN.main", "--cfg", f"configs/thesis/AML-{DATASET}-{model}.yaml",
           "--gpu", "0", "seed", str(SEED), "out_dir", OUT_DIR, "dataset.dir", DATA_DIR,
           "name_tag", TAG, *OVERRIDES]
    print(f"[GPU fisik {gpu}] " + " ".join(cmd))
    log = open(f"{OUT_DIR}/stdout_{model}_gpu{gpu}.txt", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT), log


def last_status(model, gpu):
    path = f"{run_dir_of(model, gpu)}/logging.log"
    if not os.path.exists(path):
        return "(belum mulai / memuat data)"
    lines = [l.strip() for l in open(path) if l.startswith(("> Epoch", "Early stopping", "Best epoch"))]
    return lines[-1] if lines else "(memuat data / epoch pertama)"


def wait_all(procs):
    t0 = time.time()
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(600)
        print(f"--- {(time.time() - t0) / 60:.0f} menit")
        for (model, gpu) in procs:
            print(f"  {model:9s} gpu{gpu}: {last_status(model, gpu)}")
    for (model, gpu), (p, log) in procs.items():
        log.close()
        if p.returncode != 0:
            print(f"\n!!! {model} gpu{gpu} gagal (exit {p.returncode}). 40 baris terakhir:")
            print("".join(open(f"{OUT_DIR}/stdout_{model}_gpu{gpu}.txt").readlines()[-40:]))
    print(f"Selesai dalam {(time.time() - t0) / 3600:.2f} jam")


todo = [(m, g) for m, g in JOBS if not os.path.exists(f"{run_dir_of(m, g)}/run_summary.json")]
if N_GPU >= 2:
    procs = {}
    for m, g in todo:
        procs[(m, g)] = launch(m, g)
        time.sleep(60)  # selang kecil agar pemuatan data tidak berebut I/O di detik yang sama
    wait_all(procs)
else:
    for m, g in todo:
        wait_all({(m, g): launch(m, g)})

## 4. Ringkasan: dibandingkan dengan run sebelumnya (akumulasi 4, berurutan)

Periode ekor = mulai 11 Sep 2022 (detik ke-864.000 sejak transaksi pertama 1 Sep 2022 00:00).

In [ ]:
PREV = {  # uji protokol tesis LI, seed 1, batch_accumulation 4, satu run per sesi
    "Multi-GIN": {"test_f1": 22.93, "f1_aktif": 14.71, "best_epoch": 17, "epochs_run": 20, "train_epoch_s": 363},
    "MEGA-GIN": {"test_f1": 33.00, "f1_aktif": 22.72, "best_epoch": 17, "epochs_run": 20, "train_epoch_s": 352},
}
PAPER = {"Multi-GIN": 32.21, "MEGA-GIN": 43.67}
TAIL_START = 864000


def f1_at_half(y, s):
    pred = s > 0.5
    tp, fp, fn = (pred & (y == 1)).sum(), (pred & (y == 0)).sum(), (~pred & (y == 1)).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return 100 * (2 * p * r / (p + r) if p + r else 0.0)


def recall_at(y, s, frac):
    k = max(1, round(frac * len(y)))
    top = np.argsort(-s, kind="stable")[:k]
    return 100 * y[top].sum() / max(1, y.sum())


rows = []
for m, g in JOBS:
    rd = run_dir_of(m, g)
    if not os.path.exists(f"{rd}/run_summary.json"):
        print(f"{m}: belum ada hasil")
        continue
    s = json.load(open(f"{rd}/run_summary.json"))
    d = np.load(f"{rd}/scores_test.npz")
    y, sc, ts = d["y"].astype(int), d["score"], d["timestamp"]
    act = ts < TAIL_START
    rows.append({
        "model": m, "gpu": g, "epochs_run": s["epochs_run"], "best_epoch": s["best_epoch"],
        "early_stop": s["stopped_early"],
        "val_f1": 100 * s["val"]["f1"], "test_f1": 100 * s["test"]["f1"],
        "test_f1_sebelum": PREV[m]["test_f1"], "paper_f1": PAPER[m],
        "f1_aktif": f1_at_half(y[act], sc[act]), "f1_aktif_sebelum": PREV[m]["f1_aktif"],
        "recall@0.1%_aktif": recall_at(y[act], sc[act], 0.001),
        "recall@0.5%_aktif": recall_at(y[act], sc[act], 0.005),
        "train_epoch_s": np.mean(s["train_epoch_s"]), "train_epoch_s_berurutan": PREV[m]["train_epoch_s"],
        "val_eval_s": np.mean(s["val_eval_s"]),
        "run_jam": (sum(s["train_epoch_s"]) + sum(s["val_eval_s"]) + s["test_eval_s"]) / 3600,
        "gpu_mem_gb": s["peak_gpu_mem_reserved_mb"] / 1024,
        "val_f1_per_epoch": [round(100 * r["f1"], 1) for r in map(json.loads, open(f"{rd}/val/stats.json"))],
    })

if rows:
    res = pd.DataFrame(rows)
    display(res.drop(columns=["val_f1_per_epoch"]).round(2).T)
    for r in rows:
        print(f"{r['model']:9s} F1 val per epoch: {r['val_f1_per_epoch']}")
    slow = res["train_epoch_s"] / res["train_epoch_s_berurutan"]
    print(f"\nWaktu latih per epoch paralel / berurutan: {', '.join(f'{x:.2f}x' for x in slow)} "
          f"-> perkiraan penghematan kuota dengan 2 GPU: {2 / slow.mean():.2f}x lebih banyak run per jam sesi")

## 5. Metadata analisis (dengan log lengkap)

Dijalankan setelah training agar tidak mengganggu pengukuran waktu. Output tiap dataset juga disimpan di `meta_<dataset>.log`.

In [ ]:
for name in DATASETS:
    out = f"{OUT_DIR}/meta_{name}.csv.gz"
    pat = glob.glob(f"/kaggle/input/**/{PATTERNS[name]}", recursive=True)
    print(f"===== {name} | raw: {RAW.get(name)} | patterns: {pat}")
    if name not in RAW or not pat:
        print("Lewati: file mentah atau file pola tidak ditemukan")
        continue
    with open(f"{OUT_DIR}/meta_{name}.log", "w") as log:
        p = subprocess.run(["python", "thesis/metadata.py", "--trans", RAW[name], "--patterns", pat[0], "--out", out],
                           cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT)
    print(open(f"{OUT_DIR}/meta_{name}.log").read()[-4000:])
    print(f"exit code {p.returncode} | file ada: {os.path.exists(out)}")

In [ ]:
!cd /kaggle/working && rm -f results_diag.zip && zip -qr results_diag.zip results_diag -x "*/ckpt/*" && ls -lh results_diag.zip